In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
!pip install -q -U torch torchvision torchaudio

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen1.5-1.8B-Chat"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
def generate_summary_and_recommendation(reviews_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "Anda adalah sistem analitik AI. Tugas Anda adalah merangkum "
                "ulasan negatif mahasiswa terkait kegiatan PKKMB dan menyusun "
                "rekomendasi perbaikan yang konstruktif dan terarah. "
                "Format keluaran harus terdiri dari:\n"
                "1. Ringkasan Ulasan Negatif\n"
                "2. Rekomendasi Tindak Lanjut"
            )
        },
        {
            "role": "user",
            "content": f"Kumpulan ulasan negatif mahasiswa:\n{reviews_text}\n\nLakukan perangkuman dan berikan rekomendasi."
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.3,
        top_p=0.9,
        do_sample=True
    )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
    return response

In [ ]:
import pandas as pd

direktori_dataset = 'Sentimen_Cleaned.csv'
df = pd.read_csv(direktori_dataset)

df_negatif = df[df['Sentimen'].str.lower() == 'negatif']

sampel_ulasan_negatif = df_negatif['Kritik dan saran'].dropna().head(15).tolist()
teks_input_ulasan = "\n".join([f"- {ulasan}" for ulasan in sampel_ulasan_negatif])

hasil_inferensi = generate_summary_and_recommendation(teks_input_ulasan)
print(hasil_inferensi)

In [ ]:
%%writefile ai_summarizer.py
import torch

class ReviewSummarizer:
    def __init__(self, model_id: str = "Qwen/Qwen1.5-1.8B-Chat"):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )

    def generate_summary_and_recommendation(self, reviews_text: str) -> str:
        messages = [
            {
                "role": "system",
                "content": (
                    "Anda adalah sistem analitik AI. Tugas Anda adalah merangkum "
                    "ulasan negatif mahasiswa terkait kegiatan PKKMB dan menyusun "
                    "rekomendasi perbaikan yang konstruktif dan terarah. "
                    "Format keluaran harus terdiri dari:\n"
                    "1. Ringkasan Ulasan Negatif\n"
                    "2. Rekomendasi Tindak Lanjut"
                )
            },
            {
                "role": "user",
                "content": f"Kumpulan ulasan negatif mahasiswa:\n{reviews_text}\n\nLakukan perangkuman dan berikan rekomendasi."
            }
        ]

        prompt_text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([prompt_text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.3,
            top_p=0.9,
            do_sample=True
        )

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        response = self.tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]
        return response

In [ ]:
import importlib
import ai_summarizer

importlib.reload(ai_summarizer)
from ai_summarizer import ReviewSummarizer

summarizer = ReviewSummarizer()
hasil = summarizer.generate_summary_and_recommendation(teks_input_ulasan)

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from ai_summarizer import ReviewSummarizer

app = FastAPI(title="API Evaluasi Sentimen PKKMB")
summarizer = ReviewSummarizer()

class ReviewRequest(BaseModel):
    reviews_text: str

class ReviewResponse(BaseModel):
    hasil_inferensi: str

@app.post("/api/summarize", response_model=ReviewResponse)
async def summarize_reviews(request: ReviewRequest):
    try:
        hasil = summarizer.generate_summary_and_recommendation(request.reviews_text)
        return ReviewResponse(hasil_inferensi=hasil)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import subprocess
import time
import requests
import json

server_process = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"]
)

time.sleep(30)

url = "http://127.0.0.1:8000/api/summarize"
headers = {"Content-Type": "application/json"}
payload = {
    "reviews_text": "- Server sering down dan sulit diakses saat mengerjakan tugas.\n- UI/UX website membingungkan untuk mahasiswa baru."
}

start_time = time.time()
try:
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    latency = time.time() - start_time

    if response.status_code == 200:
        hasil_json = response.json()
        print(f"Status HTTP: {response.status_code}")
        print(f"Latensi Inferensi: {latency:.2f} detik")
        print(f"Respons JSON:\n{json.dumps(hasil_json, indent=2)}")
    else:
        print(f"Galat HTTP {response.status_code}: {response.text}")
except requests.exceptions.ConnectionError:
    print("Koneksi ditolak. Lakukan verifikasi manual terhadap log Uvicorn.")
finally:
    server_process.terminate()

In [ ]:
import nest_asyncio
import uvicorn
import threading
from main import app

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

In [ ]:
import requests
import json

url = "http://127.0.0.1:8000/api/summarize"
payload = {
    "reviews_text": "- Server sering down dan sulit diakses.\n- Respon sistem lambat."
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "ISI_TOKEN_NGROK"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000)
print(f"URL Publik untuk Tim Full-Stack: {public_url}")
print("Gunakan URL di atas sebagai BASE_URL pada file .env atau konfigurasi di Backend/Frontend.")